# High-Dimensional Vectorized Model Testing with impulse-mcmc

This notebook demonstrates the performance and capabilities of the impulse-mcmc sampler on a challenging high-dimensional inference problem. We'll implement a multivariate regression model with correlated parameters that serves as an excellent test case for parallel tempering MCMC.

## Model Description

We consider a **Bayesian linear regression** problem with the following characteristics:

- **High dimensionality**: 50 regression coefficients
- **Correlated structure**: Parameters have non-trivial correlations
- **Realistic noise**: Heteroscedastic errors
- **Informative priors**: Hierarchical structure with hyperparameters

### Mathematical Formulation

**Likelihood:**
$$y_i \sim \mathcal{N}(\mathbf{x}_i^T \boldsymbol{\beta}, \sigma^2)$$

**Priors:**
- $\boldsymbol{\beta} \sim \mathcal{N}(\mathbf{0}, \tau^2 \mathbf{C})$ where $\mathbf{C}$ is a correlation matrix
- $\tau \sim \text{Half-Cauchy}(0, 2.5)$ (scale parameter)
- $\sigma \sim \text{Half-Cauchy}(0, 1)$ (noise parameter)

This creates a **52-dimensional parameter space** that tests the sampler's ability to handle:
1. High-dimensional spaces
2. Strong parameter correlations  
3. Mixed continuous parameters
4. Realistic computational demands

In [ ]:
!pip install seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.linalg import cholesky
import time

# Import the impulse sampler
from impulse import PTSampler, effective_sample_size, grubin

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## Data Generation

We'll generate synthetic data from our model to create a controlled test case where we know the true parameter values. This allows us to assess the sampler's accuracy and convergence properties.

In [ ]:
# Model dimensions
n_obs = 200      # Number of observations
n_features = 50  # Number of regression coefficients
n_params = n_features + 2  # beta coefficients + tau + sigma

print(f"Problem dimensions:")
print(f"  Observations: {n_obs}")
print(f"  Features: {n_features}")
print(f"  Total parameters: {n_params}")

# Generate correlated design matrix
# Create correlation structure in predictors
rho = 0.3  # Base correlation
X_cov = np.full((n_features, n_features), rho)
np.fill_diagonal(X_cov, 1.0)

# Generate correlated predictors
X = np.random.multivariate_normal(np.zeros(n_features), X_cov, size=n_obs)
X = stats.zscore(X, axis=0)  # Standardize

# True parameter values
true_tau = 1.5
true_sigma = 0.8

# Generate true coefficients with correlation structure
# Create correlation matrix for coefficients
C = np.exp(-0.1 * np.abs(np.subtract.outer(np.arange(n_features), np.arange(n_features))))
true_beta = np.random.multivariate_normal(np.zeros(n_features), (true_tau**2) * C)

# Generate observations
y_mean = X @ true_beta
y = np.random.normal(y_mean, true_sigma)

print(f"\nData summary:")
print(f"  y range: [{y.min():.2f}, {y.max():.2f}]")
print(f"  True tau: {true_tau:.2f}")
print(f"  True sigma: {true_sigma:.2f}")
print(f"  True beta range: [{true_beta.min():.2f}, {true_beta.max():.2f}]")

## Vectorized Model Implementation

Here we implement the **vectorized likelihood and prior functions** that can efficiently evaluate multiple parameter vectors simultaneously. This is crucial for performance with the parallel tempering sampler.

### Parameter Ordering
- `params[0:50]`: Regression coefficients $\boldsymbol{\beta}$
- `params[50]`: Scale parameter $\tau$ (log-transformed)
- `params[51]`: Noise parameter $\sigma$ (log-transformed)

We use log-transformations for positive parameters to ensure they remain positive during sampling.

In [ ]:
# Precompute matrices for efficiency
XtX = X.T @ X
Xty = X.T @ y
yty = y.T @ y
C_inv = np.linalg.inv(C)
log_det_C = np.linalg.slogdet(C)[1]

def vectorized_log_likelihood(params_array):
    """
    Vectorized log-likelihood function for Bayesian linear regression.
    
    Parameters
    ----------
    params_array : np.ndarray, shape (n_samples, n_params)
        Each row contains [beta_0, ..., beta_49, log_tau, log_sigma]
    
    Returns
    -------
    log_likelihood : np.ndarray, shape (n_samples,)
        Log-likelihood for each parameter vector
    """
    n_samples = params_array.shape[0]
    
    # Extract parameters
    beta = params_array[:, :n_features]  # (n_samples, n_features)
    log_sigma = params_array[:, -1]      # (n_samples,)
    sigma = np.exp(log_sigma)
    
    # Vectorized residual computation
    # residuals = y - X @ beta.T  # (n_obs, n_samples)
    # More efficient using precomputed matrices:
    
    # Compute sum of squared residuals for each sample
    # ||y - X*beta||^2 = y'y - 2*beta'*X'*y + beta'*X'*X*beta
    beta_Xty = np.sum(beta * Xty[None, :], axis=1)  # (n_samples,)
    beta_XtX_beta = np.sum(beta * (beta @ XtX.T), axis=1)  # (n_samples,)
    
    sse = yty - 2 * beta_Xty + beta_XtX_beta  # (n_samples,)
    
    # Log-likelihood computation
    log_likelihood = (
        -0.5 * n_obs * np.log(2 * np.pi)
        - n_obs * log_sigma
        - 0.5 * sse / (sigma**2)
    )
    
    return log_likelihood


def vectorized_log_prior(params_array):
    """
    Vectorized log-prior function.
    
    Parameters
    ----------
    params_array : np.ndarray, shape (n_samples, n_params)
    
    Returns
    -------
    log_prior : np.ndarray, shape (n_samples,)
    """
    n_samples = params_array.shape[0]
    
    # Extract parameters
    beta = params_array[:, :n_features]     # (n_samples, n_features)
    log_tau = params_array[:, -2]           # (n_samples,)
    log_sigma = params_array[:, -1]         # (n_samples,)
    
    tau = np.exp(log_tau)
    sigma = np.exp(log_sigma)
    
    # Prior for beta: N(0, tau^2 * C)
    # log p(beta | tau) = -0.5 * beta' * C_inv * beta / tau^2 - 0.5 * log|tau^2 * C|
    beta_C_inv_beta = np.sum(beta * (beta @ C_inv.T), axis=1)  # (n_samples,)
    
    log_p_beta = (
        -0.5 * n_features * np.log(2 * np.pi)
        - n_features * log_tau
        - 0.5 * log_det_C
        - 0.5 * beta_C_inv_beta / (tau**2)
    )
    
    # Half-Cauchy prior for tau: p(tau) = 2/(pi * 2.5) * 1/(1 + (tau/2.5)^2)
    log_p_tau = (
        np.log(2.0) - np.log(np.pi) - np.log(2.5)
        - np.log(1 + (tau / 2.5)**2)
        + log_tau  # Jacobian for log-transform
    )
    
    # Half-Cauchy prior for sigma: p(sigma) = 2/(pi * 1) * 1/(1 + sigma^2)
    log_p_sigma = (
        np.log(2.0) - np.log(np.pi)
        - np.log(1 + sigma**2)
        + log_sigma  # Jacobian for log-transform
    )
    
    return log_p_beta + log_p_tau + log_p_sigma


# Test the functions with a single parameter vector
test_params = np.concatenate([true_beta, [np.log(true_tau)], [np.log(true_sigma)]])
test_array = test_params[None, :]  # Add batch dimension

test_ll = vectorized_log_likelihood(test_array)[0]
test_lp = vectorized_log_prior(test_array)[0]

print(f"Test evaluation at true parameters:")
print(f"  Log-likelihood: {test_ll:.3f}")
print(f"  Log-prior: {test_lp:.3f}")
print(f"  Log-posterior: {test_ll + test_lp:.3f}")

## Sampler Configuration

We'll configure the parallel tempering sampler with:

1. **Multiple proposal types**: Adaptive Metropolis (AM), Single Component AM (SCAM), and Differential Evolution (DE)
2. **Temperature ladder**: Optimized for this problem's complexity
3. **Vectorized evaluation**: For computational efficiency
4. **Comprehensive diagnostics**: To monitor convergence

The combination of different proposal mechanisms helps the sampler adapt to the correlation structure in our parameter space.

In [ ]:
# Configure proposal bundle
# Use a mix of adaptive proposals for high-dimensional sampling
amweight = 0.2
scamweight = 0.6
deweight = 0.2

# Create initial parameter vector
# Start from reasonable initial values
initial_params = np.concatenate([
    np.zeros(n_features),     # Start beta at zero
    [np.log(1.0)],           # Start tau at 1.0 (log(1) = 0)
    [np.log(1.0)]            # Start sigma at 1.0 (log(1) = 0)
])

print(f"Sampler configuration:")
print(f"  Dimensions: {n_params}")
print(f"  Proposals: AM (20%), SCAM (60%), DE (20%)")
print(f"  Vectorized: True")
print(f"  Initial log_tau: {initial_params[-2]:.2f}")
print(f"  Initial log_sigma: {initial_params[-1]:.2f}")

# Create the sampler
sampler = PTSampler(
    ndim=n_params,
    lnlike=vectorized_log_likelihood,
    lnprior=vectorized_log_prior,
    ntemps=30,  # Use multiple temperatures for this challenging problem
    am_weight=amweight,
    scam_weight=scamweight,
    de_weight=deweight,
    vectorized=True,
    seed=42
)

print(f"\nPTSampler created with {sampler.ntemps} temperature chains")

## Running the Sampler

Now we'll run the parallel tempering sampler on our high-dimensional problem. We'll monitor:

- **Sampling progress** with timing information
- **Acceptance rates** across temperature chains
- **Temperature swapping** efficiency
- **Real-time convergence** diagnostics

This serves as both a demonstration and a stress test of the sampler's capabilities.

In [ ]:
# Sampling parameters
n_iterations = 50_000
n_burnin = 5000

print(f"Starting sampling with {n_iterations} iterations...")
print(f"Burn-in: {n_burnin} iterations")
print(f"Effective samples: {n_iterations - n_burnin} iterations")

# Time the sampling
start_time = time.time()

# Run the sampler
sampler.sample(
    initial_params,
    num_iterations=n_iterations
)

end_time = time.time()
sampling_time = end_time - start_time

print(f"\nSampling completed!")
print(f"Total time: {sampling_time:.2f} seconds")
print(f"Time per iteration: {sampling_time/n_iterations*1000:.2f} ms")
print(f"Effective samples per second: {(n_iterations - n_burnin)/sampling_time:.1f}")

## Convergence Diagnostics

Let's examine the sampler's performance and convergence properties. We'll analyze:

1. **Acceptance rates** across temperature chains
2. **Temperature swap statistics**
3. **Trace plots** for key parameters
4. **Effective sample sizes**
5. **Parameter recovery** accuracy

In [ ]:
for i in range(sampler.ndim):
    plt.plot(chain[:, 0, 0])
    plt.axhline(true_beta[0], color='r', linestyle='--')

In [ ]:
print(sampler.ntemps)

In [ ]:
chain_data = sampler.load_chain()
chain = chain_data['samples']  # shape (ntemps, nsamples, ndim)
print(f"Chain shape: {chain.shape}")

In [ ]:
# Extract chains (remove burn-in)
chains = chain[0, n_burnin:, :]  # Coldest chain (temperature index 0), post burn-in
n_samples = chains.shape[0]

print(f"Post-burnin analysis:")
print(f"  Chain shape: {chains.shape}")
print(f"  Effective samples: {n_samples}")

# Extract parameter chains
beta_chains = chains[:, :n_features]
tau_chains = np.exp(chains[:, -2])  # Transform back from log
sigma_chains = np.exp(chains[:, -1])  # Transform back from log

# Compute posterior summaries
beta_mean = np.mean(beta_chains, axis=0)
beta_std = np.std(beta_chains, axis=0)
tau_mean = np.mean(tau_chains)
tau_std = np.std(tau_chains)
sigma_mean = np.mean(sigma_chains)
sigma_std = np.std(sigma_chains)

print(f"\nPosterior summaries:")
print(f"  tau: {tau_mean:.3f} ± {tau_std:.3f} (true: {true_tau:.3f})")
print(f"  sigma: {sigma_mean:.3f} ± {sigma_std:.3f} (true: {true_sigma:.3f})")
print(f"  beta mean error: {np.mean(np.abs(beta_mean - true_beta)):.4f}")
print(f"  beta RMSE: {np.sqrt(np.mean((beta_mean - true_beta)**2)):.4f}")

# Coverage analysis for beta parameters
beta_lower = np.percentile(beta_chains, 2.5, axis=0)
beta_upper = np.percentile(beta_chains, 97.5, axis=0)
coverage = np.mean((true_beta >= beta_lower) & (true_beta <= beta_upper))
print(f"  95% credible interval coverage: {coverage:.1%}")

In [ ]:
# Acceptance rate analysis
if hasattr(sampler, 'acceptance_rates'):
    print(f"\nAcceptance rates by temperature:")
    for i, rate in enumerate(sampler.acceptance_rates):
        print(f"  Chain {i}: {rate:.1%}")

# Temperature swap analysis
if hasattr(sampler, 'swap_acceptance_rate'):
    print(f"\nTemperature swap acceptance: {sampler.swap_acceptance_rate:.1%}")

# Create diagnostic plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Trace plots for hyperparameters
axes[0, 0].plot(tau_chains)
axes[0, 0].axhline(true_tau, color='red', linestyle='--', label='True value')
axes[0, 0].set_title('Trace: τ (scale parameter)')
axes[0, 0].set_ylabel('τ')
axes[0, 0].legend()

axes[0, 1].plot(sigma_chains)
axes[0, 1].axhline(true_sigma, color='red', linestyle='--', label='True value')
axes[0, 1].set_title('Trace: σ (noise parameter)')
axes[0, 1].set_ylabel('σ')
axes[0, 1].legend()

# Sample beta traces (first 5 coefficients)
for i in range(5):
    axes[0, 2].plot(beta_chains[:, i], alpha=0.7, label=f'β{i}')
axes[0, 2].set_title('Trace: First 5 β coefficients')
axes[0, 2].set_ylabel('β')
axes[0, 2].legend()

# Posterior distributions
axes[1, 0].hist(tau_chains, bins=50, alpha=0.7, density=True)
axes[1, 0].axvline(true_tau, color='red', linestyle='--', label='True value')
axes[1, 0].set_title('Posterior: τ')
axes[1, 0].set_xlabel('τ')
axes[1, 0].legend()

axes[1, 1].hist(sigma_chains, bins=50, alpha=0.7, density=True)
axes[1, 1].axvline(true_sigma, color='red', linestyle='--', label='True value')
axes[1, 1].set_title('Posterior: σ')
axes[1, 1].set_xlabel('σ')
axes[1, 1].legend()

# Beta recovery plot
axes[1, 2].scatter(true_beta, beta_mean, alpha=0.6)
axes[1, 2].plot([-3, 3], [-3, 3], 'r--', label='Perfect recovery')
axes[1, 2].set_xlabel('True β values')
axes[1, 2].set_ylabel('Posterior mean β')
axes[1, 2].set_title('Parameter Recovery')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Performance Analysis

Let's analyze the computational performance and scalability of the vectorized implementation compared to a hypothetical scalar version.

In [ ]:
# Performance comparison: vectorized vs scalar evaluation
n_test_samples = 1000
test_params_array = np.random.randn(n_test_samples, n_params)

# Time vectorized evaluation
start = time.time()
vec_ll = vectorized_log_likelihood(test_params_array)
vec_lp = vectorized_log_prior(test_params_array)
vec_time = time.time() - start

# Simulate scalar evaluation timing
start = time.time()
for i in range(100):  # Scale down for timing
    _ = vectorized_log_likelihood(test_params_array[i:i+1])
    _ = vectorized_log_prior(test_params_array[i:i+1])
scalar_time_scaled = (time.time() - start) * 10  # Scale up

speedup = scalar_time_scaled / vec_time

print(f"Performance Analysis:")
print(f"  Vectorized evaluation ({n_test_samples} samples): {vec_time*1000:.2f} ms")
print(f"  Estimated scalar time: {scalar_time_scaled*1000:.2f} ms")
print(f"  Speedup factor: {speedup:.1f}x")
print(f"  Evaluations per second (vectorized): {n_test_samples/vec_time:.0f}")

# Memory efficiency analysis
import sys
param_size = test_params_array.nbytes
result_size = vec_ll.nbytes + vec_lp.nbytes

print(f"\nMemory Usage:")
print(f"  Parameter array: {param_size/1024:.1f} KB")
print(f"  Result arrays: {result_size/1024:.1f} KB")
print(f"  Memory efficiency: {param_size/result_size:.1f}:1 (input:output)")

## Advanced Diagnostics

Let's perform more sophisticated convergence diagnostics to thoroughly validate our results.

In [ ]:
# Compute effective sample sizes using impulse diagnostics
ess_all = effective_sample_size(chains)  # shape (n_params+extra,)

ess_tau = ess_all[n_features]  # log_tau column
ess_sigma = ess_all[n_features + 1]  # log_sigma column
ess_beta = ess_all[:n_features]

print(f"Effective Sample Sizes:")
print(f"  τ: {ess_tau:.0f} ({ess_tau/n_samples:.1%} efficiency)")
print(f"  σ: {ess_sigma:.0f} ({ess_sigma/n_samples:.1%} efficiency)")
print(f"  β (mean): {np.nanmean(ess_beta):.0f} ({np.nanmean(ess_beta)/n_samples:.1%} efficiency)")
print(f"  β (min): {np.nanmin(ess_beta):.0f} ({np.nanmin(ess_beta)/n_samples:.1%} efficiency)")

# Gelman-Rubin R-hat using impulse diagnostics
rhat, rhat_flagged = grubin(chains)
print(f"\nGelman-Rubin R-hat (should be < 1.01):")
print(f"  τ: {rhat[n_features]:.3f}")
print(f"  σ: {rhat[n_features + 1]:.3f}")
print(f"  β (max): {np.nanmax(rhat[:n_features]):.3f}")

# Summary of sampler performance
print(f"\n" + "="*50)
print(f"SAMPLER PERFORMANCE SUMMARY")
print(f"="*50)
print(f"Problem complexity: {n_params}D parameter space")
print(f"Sampling efficiency: {np.nanmean(ess_beta)/n_samples:.1%} (mean ESS/samples)")
print(f"Parameter recovery: {coverage:.1%} credible interval coverage")
print(f"Computational speed: {(n_iterations - n_burnin)/sampling_time:.0f} effective samples/sec")

## Conclusion

This notebook demonstrates the **impulse-mcmc** sampler's capabilities on a challenging **52-dimensional Bayesian inference problem**. Key achievements:

### ✅ **Technical Validation**
- **High-dimensional sampling**: Successfully handled 50+ correlated parameters
- **Vectorized efficiency**: Significant computational speedup through vectorization
- **Robust convergence**: Effective sample sizes and parameter recovery
- **Adaptive proposals**: Multiple proposal types working in concert

### ✅ **Practical Benefits**
- **Real-world complexity**: Hierarchical model with realistic correlation structure
- **Computational efficiency**: Fast evaluation suitable for production use
- **Comprehensive diagnostics**: Built-in monitoring and validation tools
- **Flexible interface**: Easy to adapt to different model structures

### 🎯 **Use Cases**
This test case demonstrates the sampler's suitability for:
- **High-dimensional regression** problems
- **Hierarchical Bayesian models** with hyperparameters
- **Correlated parameter spaces** requiring adaptive proposals
- **Production inference** where computational efficiency matters

The **parallel tempering** approach with **vectorized likelihood evaluation** provides an excellent balance of exploration capability and computational efficiency for complex inference problems.